# Emotion Recognition API — Google Colab
FastAPI + ModelScope `emotion2vec_base_finetuned` + ngrok public tunnel.

**Run each cell top-to-bottom. Set your ngrok authtoken in the last cell before starting the server.**

## Step 1 — Install Required Packages

In [ ]:
# Core dependencies
!pip install fastapi uvicorn nest-asyncio pyngrok soundfile librosa pydub

# ModelScope + FunASR (emotion2vec lives here)
!pip install modelscope funasr
!pip install "modelscope[audio]" -f https://modelscope.oss-cn-beijing.aliyuncs.com/releases/repo.html

print("✅ All packages installed.")

: 

## Step 2 — Import Libraries & Configure Logging

In [ ]:
import nest_asyncio
import threading
import uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

nest_asyncio.apply()

print("✅ Libraries imported.")

## Step 3 — Load the Emotion Recognition Model
Uses `iic/emotion2vec_base_finetuned` from ModelScope. First run will download ~300 MB.

In [ ]:
from modelscope.pipelines import pipeline

print("Loading model...")
model = pipeline(
    task="emotion-recognition",
    model="iic/emotion2vec_base_finetuned",
    model_revision="v2.0.4",
)
print("Model loaded successfully!")

## Step 4 — Initialize FastAPI Application

In [ ]:
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("✅ FastAPI app created.")

## Step 5 — Register Endpoints

`GET /health` — liveness check.  
`POST /emotion_recognition` — upload an audio file, get emotion scores back.  
`POST /predict` — alias for `/emotion_recognition`.

In [ ]:
import os, tempfile
from fastapi import HTTPException
from fastapi.responses import JSONResponse

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.post("/emotion_recognition")
async def emotion_recognition(audio: UploadFile = File(...)):
    try:
        audio_bytes = await audio.read()
        if len(audio_bytes) < 512:
            return JSONResponse(status_code=400, content={"error": "Audio too small or empty"})

        # Use the real content-type to pick the right extension
        ct = (audio.content_type or "audio/webm").lower()
        if "mp4" in ct:   ext = ".mp4"
        elif "ogg" in ct: ext = ".ogg"
        elif "wav" in ct: ext = ".wav"
        else:             ext = ".webm"   # default — browser MediaRecorder output

        tmp_path = f"/tmp/sos_audio{ext}"
        with open(tmp_path, "wb") as f:
            f.write(audio_bytes)

        print(f"Running model on {tmp_path}  ({len(audio_bytes)} bytes, ct={ct})")
        result = model(tmp_path)
        print(f"Result: {result}")

        try:
            os.remove(tmp_path)
        except Exception:
            pass

        return result

    except Exception as e:
        import traceback
        traceback.print_exc()
        return JSONResponse(status_code=500, content={"error": str(e)})

@app.post("/predict")
async def predict(audio: UploadFile = File(...)):
    return await emotion_recognition(audio)

print("✅ Endpoints registered.")

## Step 6 — Start Server with Ngrok

Kills any existing ngrok tunnel, starts a new one, then launches uvicorn on a background thread  
(works in both Colab and Jupyter without blocking the kernel).  
Replace `NGROK_AUTHTOKEN` with your token from https://ngrok.com/ if needed.

In [ ]:
NGROK_AUTHTOKEN = "2yHTje3txqoaytgEfiju1cH48QG_42Mo5txeB7MPfjexnWk2x"  # ← your token
PORT = 8000

def start_server():
    nest_asyncio.apply()

    # Kill any existing ngrok tunnel before starting a new one
    ngrok.kill()

    if NGROK_AUTHTOKEN:
        ngrok.set_auth_token(NGROK_AUTHTOKEN)
        print("\n🔌 Starting ngrok tunnel...")
        public_url = ngrok.connect(PORT)
        print(f"🌍 PUBLIC URL: {public_url}")
        print(f"🎯 Emotion endpoint: {public_url}/emotion_recognition")
        print(f"📖 Swagger docs:     {public_url}/docs")
    else:
        print(f"ℹ️  No ngrok token — running locally only.")

    print(f"📝 Local Swagger UI: http://localhost:{PORT}/docs")

    config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
    server = uvicorn.Server(config)

    thread = threading.Thread(target=server.run, daemon=True)
    thread.start()

start_server()